# 🤖 Multi-Document Conversational RAG Chatbot (Enterprise RAG)

This notebook builds a hybrid AI chatbot using **local HuggingFace embeddings** and **OpenAI's gpt-4o-mini** for the Chatbot Brain.

---

## ⚠️ IMPORTANT: Enable GPU First!
Go to **Runtime → Change runtime type → GPU (T4)** before running any cells.

## Step 1a: Install Dependencies

Run this cell, then **restart the runtime** (Runtime → Restart runtime) before continuing to Step 1b.

In [ ]:
import numpy
NV = numpy.__version__
print(f'🔒 Locking numpy to: {NV}')

%pip install -q "numpy=={NV}" \
    "langchain<1.0" "langchain-core<1.0" langchain-community langchain-huggingface langchain-chroma langchain-openai \
    chromadb "sentence-transformers==2.7.0" "transformers==4.41.2" accelerate gradio

print('\n✅ Install complete!')
print('⚠️ NOW GO TO: Runtime → Restart runtime')
print('   Then run Step 1b below.')

## Step 1b: Verify Installation

Run this **after restarting the runtime** to confirm everything works.

In [ ]:
from langchain.chains import create_retrieval_chain
from sentence_transformers import SentenceTransformer
import transformers, gradio
print(f'transformers: {transformers.__version__}')
print(f'gradio: {gradio.__version__}')
print('✅ All packages verified! Proceed to Step 2.')

## Step 2.1: Create Folder Structure

In [ ]:
import os

folders = ['features', 'integrations', 'billing', 'support']
for f in folders:
    os.makedirs(f, exist_ok=True)
    print(f'✅ Created folder: {f}/')

print('\n📁 Folders are ready! Now run Step 2.2 to upload your files.')

## Step 2.2: Upload Your `.md` Files

**Repeat this cell** for each folder (run it 4 times, once per folder).

In [ ]:
from google.colab import files
import os

print('Select which folder to upload files into:')
print('1. features')
print('2. integrations')
print('3. billing')
print('4. support')

choice = input('\nEnter folder number (1-4): ')
folder_map = {'1': 'features', '2': 'integrations', '3': 'billing', '4': 'support'}
target_folder = folder_map.get(choice, 'features')

print(f'\n📂 Upload your .md files for the "{target_folder}" folder:')
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = os.path.join(target_folder, filename)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  ✅ Saved: {dest}')

print(f'\n🎉 Uploaded {len(uploaded)} file(s) to {target_folder}/')

## Step 2.3: Verify Uploaded Files

In [ ]:
import os

total = 0
for folder in ['features', 'integrations', 'billing', 'support']:
    if os.path.exists(folder):
        file_list = [f for f in os.listdir(folder) if f.endswith('.md')]
        total += len(file_list)
        print(f'📂 {folder}: {len(file_list)} file(s) → {file_list}')
    else:
        print(f'❌ {folder}: not found')

print(f'\n📄 Total files across all folders: {total}')
if total == 0:
    print('⚠️ No files found! Go back to Step 2.2 and upload your .md files.')
else:
    print('✅ All good! Proceed to Step 3.')

## Step 3: Build the RAG Pipeline

Loads documents, chunks them, embeds with `BAAI/bge-m3`, and stores in ChromaDB.

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

folders = ['./features', './integrations', './billing', './support']
all_documents = []

for folder_path in folders:
    if not os.path.exists(folder_path):
        continue
    loader = DirectoryLoader(
        folder_path, glob='**/*.md',
        loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'}
    )
    docs = loader.load()
    if len(docs) > 0:
        print(f'✅ Loaded {len(docs)} document(s) from {folder_path}')
    all_documents.extend(docs)

print(f'\n📄 Total documents loaded: {len(all_documents)}')

if len(all_documents) == 0:
    print('\n❌ ERROR: No documents found! Go back to Step 2 and upload your .md files.')
else:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=['\n## ', '\n### ', '\n\n', '\n', ' ', '']
    )
    chunks = text_splitter.split_documents(all_documents)
    print(f'✂️ Created {len(chunks)} chunks')

    print('\n🧠 Loading BAAI/bge-m3 embedding model (this may take 1-2 minutes)...')
    embeddings = HuggingFaceEmbeddings(
        model_name='BAAI/bge-m3',
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )
    print('✅ Embedding model loaded on GPU!')

    persist_directory = './chroma_db_colab'
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    print(f'\n🗄️ Vector store created at {persist_directory} with {len(chunks)} chunks!')

## Step 4.1: Enter your API Key

In [ ]:
import os
import getpass

print("Please enter your OpenAI API Key (it will be hidden as you type):")
os.environ["OPENAI_API_KEY"] = getpass.getpass()
print("✅ API Key saved securely in environment!")

## Step 4.2: Load the OpenAI LLM

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
print("✅ OpenAI LLM initialized successfully!")

## Step 5: Test the RAG Chain

Quick test to make sure everything works before launching the UI.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

vectorstore = Chroma(persist_directory='./chroma_db_colab', embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

prompt = ChatPromptTemplate.from_template("""You are a highly knowledgeable and helpful AI assistant.

CRITICAL INSTRUCTIONS FOR RESPONDING:
1. GENERAL GREETINGS & SELF-IDENTIFICATION ONLY:
   - If the user's query is a general greeting, simple small talk, or a question about your identity/capabilities, answer friendly and politely as an AI assistant. You do not need to use the context for these casual conversational queries.
   
2. ALL OTHER FACTUAL, TECHNICAL, AND TOPIC QUESTIONS:
   - For ALL other queries (including questions about specific documents, tools, deals, or any other factual, general knowledge, or technical questions):
     - You MUST answer the question based strictly and ONLY on the retrieved context below.
     - If the retrieved context does not contain the answer, or does not mention the topic of the question, you MUST politely state that you do not know.

RETIREVED CONTEXT:
{context}

Question:
{input}""")

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

query = 'What fields are required to create a task?'
print(f'❓ Question: {query}\n')

response = rag_chain.invoke({'input': query})
print(f'🤖 Answer:\n{response["answer"]}')

## Step 6: Launch the Chat UI 🚀

This uses **Gradio** which automatically creates a public URL in Colab — no tunnels or signups needed!

In [ ]:
%pip install -q gradio
import os
import gradio as gr
from langchain.chains import create_retrieval_chain, create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Build the full conversational chain with expanded retrieval (k=5)
vectorstore = Chroma(persist_directory='./chroma_db_colab', embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Given a chat history and the latest user question, formulate a standalone question. Do NOT answer the question.'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

# Added strict negative constraint to prevent hallucinating outside knowledge
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 
     'You are a highly knowledgeable and helpful AI assistant.\n\n'
     'CRITICAL INSTRUCTIONS FOR RESPONDING:\n'
     '1. GENERAL GREETINGS & SELF-IDENTIFICATION ONLY:\n'
     '   - If the user\'s query is a general greeting, simple small talk, or a question about your identity/capabilities (e.g., "who are you?", "what is your job?", "how can you help me?"), answer friendly and politely as an AI assistant. You do not need to use the context for these casual conversational queries.\n'
     '2. ALL OTHER FACTUAL, TECHNICAL, AND TOPIC QUESTIONS:\n'
     '   - For ALL other queries (including questions about specific documents, tools, deals, or any other factual, general knowledge, or technical questions):\n'
     '     - You MUST answer the question based strictly and ONLY on the retrieved context below.\n'
     '     - If the retrieved context does not explicitly mention the answer to the question, you MUST say: "I do not know because it is not mentioned in the provided documents." Do NOT use outside knowledge, guess, or invent details that are not in the text.\n'
     '3. Keep your answers concise, detailed, and format with bullet points and bold key terms.\n\n'
     'Context:\n{context}'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
full_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# Chat function with history and view-source functionality
chat_history = []

def chat(message, history):
    global chat_history
    response = full_rag_chain.invoke({'input': message, 'chat_history': chat_history})
    answer = response['answer']
    
    # Extract and format sources
    sources = []
    if 'context' in response:
        for idx, doc in enumerate(response['context']):
            src_path = doc.metadata.get('source', '')
            src_path = src_path.replace("\\\\", "/").replace("\\", "/")
            path_parts = src_path.split("/")
            src_display = "/".join(path_parts[-2:]) if len(path_parts) >= 2 else os.path.basename(src_path)
            preview = doc.page_content[:150].strip().replace('\n', ' ')
            sources.append(f"• **{src_display}**: *\"{preview}...\"*")
            
    if sources:
        sources_str = "\n\n---\n🔍 **Sources used to answer:**\n" + "\n".join(sources)
        full_answer = answer + sources_str
    else:
        full_answer = answer
        
    chat_history.extend([HumanMessage(content=message), AIMessage(content=answer)])
    return full_answer

# Launch Gradio UI
demo = gr.ChatInterface(
    fn=chat,
    title='💬 Enterprise RAG Chatbot',
    description='Powered by HuggingFace Embeddings & OpenAI gpt-4o-mini',
    examples=['What AI tools are available?', 'How does the AI Copilot work?', 'What billing plans are supported?'],
    theme=gr.themes.Soft()
)
demo.launch(share=True)


## Step 7: Export & Download Your RAG Test Session Results 📝

Run this cell after you are done testing your chatbot to download all Q&As!

In [ ]:
import os

def export_test_results(filename="rag_test_results.md"):
    global chat_history
    if not chat_history:
        print("⚠️ No chat history found in this session yet. Go ask a few questions in the Gradio chat first!")
        return
        
    with open(filename, "w", encoding="utf-8") as f:
        f.write("# 📝 RAG Chatbot Accuracy Test Results\n\n")
        f.write(f"This file compiles the questions asked and answers generated during this testing session.\n\n---\n\n")
        
        for i in range(0, len(chat_history), 2):
            q_num = i // 2 + 1
            question = chat_history[i].content
            answer_with_sources = chat_history[i+1].content if i + 1 < len(chat_history) else "No answer generated."
                
            f.write(f"## ❓ Question {q_num}: {question}\n\n")
            f.write(f"### 🤖 Chatbot Response:\n{answer_with_sources}\n\n")
            f.write("---\n\n")
            print(f"✅ Processed Q{q_num}: {question[:50]}...")
            
    print(f"\n🎉 Successfully compiled {len(chat_history)//2} test cases into '{filename}'!")
    
    try:
        from google.colab import files
        files.download(filename)
        print("📥 Download started in your browser!")
    except Exception:
        print(f"💾 File saved locally in Colab at: {os.path.abspath(filename)}")

export_test_results()